# Garmin Data Exploration

## Objective

Inspect the raw Garmin activity export, identify data-quality issues, and create an initial running-only dataset for later cleaning and feature engineering.

In [1]:
from pathlib import Path

import pandas as pd

In [8]:
DATA_PATH = Path("../data/raw/run_data.csv")

df = pd.read_csv(DATA_PATH)

print("Rows:", df.shape[0])
print("Columns:", df.shape[1])

df.head()
df.columns.tolist()
df.info()

pd.set_option("display.max_columns", None)
df.head(3)

Rows: 169
Columns: 35
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 169 entries, 0 to 168
Data columns (total 35 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   Activity Type             169 non-null    object 
 1   Date                      169 non-null    object 
 2   Favorite                  169 non-null    bool   
 3   Title                     169 non-null    object 
 4   Distance                  169 non-null    float64
 5   Calories                  169 non-null    object 
 6   Time                      169 non-null    object 
 7   Avg HR                    169 non-null    object 
 8   Max HR                    169 non-null    object 
 9   Aerobic TE                169 non-null    object 
 10  Avg Run Cadence           169 non-null    object 
 11  Max Run Cadence           169 non-null    object 
 12  Avg Pace                  169 non-null    object 
 13  Best Pace                 169 non-null    o

,Activity Type,Date,Favorite,Title,Distance,Calories,Time,Avg HR,Max HR,Aerobic TE,Avg Run Cadence,Max Run Cadence,Avg Pace,Best Pace,Total Ascent,Total Descent,Avg Stride Length,Avg Vertical Ratio,Avg Vertical Oscillation,Avg Ground Contact Time,Normalized Power® (NP®),Training Stress Score®,Avg Power,Max Power,Steps,Body Battery Drain,Decompression,Best Lap Time,Number of Laps,Avg Stress,Max Stress,Moving Time,Elapsed Time,Min Elevation,Max Elevation
0,Running,2026-07-11 09:15:11,False,Toronto Running,19.00,"1,225",01:47:07,166,185,4.7,182,215,5:38,3:33,112,117,0.96,8.2,7.8,243,262,0.0,259,433,"19,564",'-23,No,00:00:01.0,20,--,--,01:46:49,01:57:28,111,182
1,Running,2026-07-09 07:40:06,False,Toronto Running,9.47,588,00:49:47,170,191,4.2,179,224,5:15,2:53,100,104,1.02,7.8,7.7,232,275,0.0,264,472,"8,744",'-18,No,00:00:19.6,22,--,--,00:48:59,01:01:14,75,81
2,Running,2026-07-08 17:36:39,False,Toronto Running,7.51,490,00:49:20,150,162,3.0,179,188,6:34,4:47,60,85,0.85,9.0,7.6,257,225,0.0,223,355,"8,816",'-8,No,00:03:16.0,8,--,--,00:49:13,00:51:47,91,118


### Inspect a few full rows

In [9]:
important_columns = [
    "Activity Type",
    "Date",
    "Distance",
    "Moving Time",
    "Elapsed Time",
    "Avg Pace",
    "Avg HR",
    "Training Stress Score®",
    "Avg Power",
]

df[important_columns].head(10)

,Activity Type,Date,Distance,Moving Time,Elapsed Time,Avg Pace,Avg HR,Training Stress Score®,Avg Power
0,Running,2026-07-11 09:15:11,19.00,01:46:49,01:57:28,5:38,166,0.0,259
1,Running,2026-07-09 07:40:06,9.47,00:48:59,01:01:14,5:15,170,0.0,264
2,Running,2026-07-08 17:36:39,7.51,00:49:13,00:51:47,6:34,150,0.0,223
3,Treadmill Running,2026-07-07 09:13:03,2.61,00:07:10,00:18:08,3:08,144,0.0,386
4,Treadmill Running,2026-07-06 07:40:51,4.27,00:32:40,00:47:16,10:57,144,0.0,130
5,Running,2026-07-05 10:22:05,17.01,01:41:37,01:51:33,5:59,157,0.0,251
6,Running,2026-07-02 08:06:30,6.93,00:35:39,00:46:01,5:10,165,0.0,273
7,Running,2026-07-01 09:04:54,13.00,01:12:00,01:25:24,5:33,170,0.0,263
8,Treadmill Running,2026-06-30 08:37:43,6.15,00:39:12,00:57:15,8:53,167,0.0,158
9,Running,2026-06-29 09:18:46,8.00,00:56:37,01:00:23,7:06,146,0.0,228


### See what activity types are included

In [10]:
df["Activity Type"].value_counts(dropna=False)

Activity Type
Running              107
Treadmill Running     62
Name: count, dtype: int64

#### Running Activity Types

The export includes the following activity types that may be relevant:

- Running
- Treadmill Running

The MVP will initially exclude treadmill running as those activities have no wind, no GPS error, no elevation, and often different pacing behaviour. 

### Create Running Dataset

In [23]:
# Create a new DataFrame containing only outdoor runs
runs = df[
    df["Activity Type"] == "Running"
].copy()

print(f"Total exported activities: {len(df)}")
print(f"Outdoor running activities: {len(runs)}")

runs.head()

Total exported activities: 169
Outdoor running activities: 107


,Activity Type,Date,Favorite,Title,Distance,Calories,Time,Avg HR,Max HR,Aerobic TE,Avg Run Cadence,Max Run Cadence,Avg Pace,Best Pace,Total Ascent,Total Descent,Avg Stride Length,Avg Vertical Ratio,Avg Vertical Oscillation,Avg Ground Contact Time,Normalized Power® (NP®),Training Stress Score®,Avg Power,Max Power,Steps,Body Battery Drain,Decompression,Best Lap Time,Number of Laps,Avg Stress,Max Stress,Moving Time,Elapsed Time,Min Elevation,Max Elevation
0,Running,2026-07-11 09:15:11,False,Toronto Running,19.00,"1,225",01:47:07,166,185,4.7,182,215,5:38,3:33,112,117,0.96,8.2,7.8,243,262,0.0,259,433,"19,564",'-23,No,00:00:01.0,20,--,--,01:46:49,01:57:28,111,182
1,Running,2026-07-09 07:40:06,False,Toronto Running,9.47,588,00:49:47,170,191,4.2,179,224,5:15,2:53,100,104,1.02,7.8,7.7,232,275,0.0,264,472,"8,744",'-18,No,00:00:19.6,22,--,--,00:48:59,01:01:14,75,81
2,Running,2026-07-08 17:36:39,False,Toronto Running,7.51,490,00:49:20,150,162,3.0,179,188,6:34,4:47,60,85,0.85,9.0,7.6,257,225,0.0,223,355,"8,816",'-8,No,00:03:16.0,8,--,--,00:49:13,00:51:47,91,118
5,Running,2026-07-05 10:22:05,False,Toronto Running,17.01,"1,084",01:41:47,157,179,4.0,180,198,5:59,4:09,109,111,0.92,8.6,7.8,250,253,0.0,251,400,"18,374",'-25,No,00:00:01.7,18,--,--,01:41:37,01:51:33,116,182
6,Running,2026-07-02 08:06:30,False,Toronto Running,6.93,414,00:35:53,165,195,4.0,182,205,5:10,3:21,80,84,1.02,8.0,7.9,236,280,0.0,273,460,"6,580",'-15,No,00:01:26.6,18,--,--,00:35:39,00:46:01,75,82


The export contains two activity types:

- Running
- Treadmill Running

The initial MVP uses only outdoor `Running` activities. Treadmill runs are excluded because treadmill pace, terrain, and environmental conditions are not directly comparable with outdoor running.

### Missing Value Summary

In [24]:
missing_summary = (
    runs.isna()
    .sum()
    .sort_values(ascending=False)
    .to_frame("missing_count")
)

missing_summary["missing_percent"] = (
    missing_summary["missing_count"] / len(runs) * 100
).round(1)

missing_summary

,missing_count,missing_percent
Activity Type,0,0.0
Decompression,0,0.0
Normalized Power® (NP®),0,0.0
Training Stress Score®,0,0.0
Avg Power,0,0.0
Max Power,0,0.0
Steps,0,0.0
Body Battery Drain,0,0.0
Best Lap Time,0,0.0
Avg Vertical Oscillation,0,0.0


#### Initial Missing Data Observations
- no missing data in current dataset!

### Inspect Important Column Formats

In [15]:
columns_to_inspect = [
    "Date",
    "Distance",
    "Moving Time",
    "Elapsed Time",
    "Avg Pace",
    "Best Pace",
    "Avg HR",
    "Avg Run Cadence",
    "Avg Stride Length",
    "Total Ascent",
]

for column in columns_to_inspect:
    print(f"\n--- {column} ---")
    print(runs[column].dropna().head(5).tolist())


--- Date ---
['2026-07-11 09:15:11', '2026-07-09 07:40:06', '2026-07-08 17:36:39', '2026-07-05 10:22:05', '2026-07-02 08:06:30']

--- Distance ---
[19.0, 9.47, 7.51, 17.01, 6.93]

--- Moving Time ---
['01:46:49', '00:48:59', '00:49:13', '01:41:37', '00:35:39']

--- Elapsed Time ---
['01:57:28', '01:01:14', '00:51:47', '01:51:33', '00:46:01']

--- Avg Pace ---
['5:38', '5:15', '6:34', '5:59', '5:10']

--- Best Pace ---
['3:33', '2:53', '4:47', '4:09', '3:21']

--- Avg HR ---
['166', '170', '150', '157', '165']

--- Avg Run Cadence ---
['182', '179', '179', '180', '182']

--- Avg Stride Length ---
['0.96', '1.02', '0.85', '0.92', '1.02']

--- Total Ascent ---
['112', '100', '60', '109', '80']


### Inspect Data Types

In [18]:
data_types = pd.DataFrame({
    "column": runs.columns,
    "data_type": runs.dtypes.astype(str).values,
    "non_null_count": runs.notna().sum().values,
    "missing_count": runs.isna().sum().values,
})

data_types

,column,data_type,non_null_count,missing_count
0,Activity Type,object,107,0
1,Date,object,107,0
2,Favorite,bool,107,0
3,Title,object,107,0
4,Distance,float64,107,0
5,Calories,object,107,0
6,Time,object,107,0
7,Avg HR,object,107,0
8,Max HR,object,107,0
9,Aerobic TE,object,107,0


### Check Dates & Duplicates

In [19]:
# Inspect date values
runs["Date"].head(10)

# Test whether pandas can parse them
parsed_dates = pd.to_datetime(
    runs["Date"],
    errors="coerce"
)

print("Dates that could not be parsed:", parsed_dates.isna().sum())
print("Earliest run:", parsed_dates.min())
print("Latest run:", parsed_dates.max())

# Check duplicate rows
print("Duplicate rows:", runs.duplicated().sum())

Dates that could not be parsed: 0
Earliest run: 2025-06-14 07:40:11
Latest run: 2026-07-11 09:15:11
Duplicate rows: 0


### Inspect Basic Numeric Distributions

In [21]:
numeric_columns = [
    "Distance",
    "Calories",
    "Avg HR",
    "Max HR",
    "Aerobic TE",
    "Avg Run Cadence",
    "Avg Stride Length",
    "Avg Vertical Ratio",
    "Avg Vertical Oscillation",
    "Avg Ground Contact Time",
    "Avg Power",
    "Total Ascent",
]

runs[numeric_columns].describe().T

,count,mean,std,min,25%,50%,75%,max
Distance,107.0,8.991776,4.550423,0.54,6.565,8.29,11.865,21.13


# Data Quality Summary

In [22]:
summary = {
    "total_exported_activities": len(df),
    "outdoor_running_activities": len(runs),
    "earliest_run": parsed_dates.min(),
    "latest_run": parsed_dates.max(),
    "unparseable_dates": int(parsed_dates.isna().sum()),
    "exact_duplicate_rows": int(runs.duplicated().sum()),
    "missing_avg_hr": int(runs["Avg HR"].isna().sum()),
    "missing_avg_power": int(runs["Avg Power"].isna().sum()),
}

summary

{'total_exported_activities': 169,
 'outdoor_running_activities': 107,
 'earliest_run': Timestamp('2025-06-14 07:40:11'),
 'latest_run': Timestamp('2026-07-11 09:15:11'),
 'unparseable_dates': 0,
 'exact_duplicate_rows': 0,
 'missing_avg_hr': 0,
 'missing_avg_power': 0}

# Initial Findings

## Dataset Coverage

- Total activities: 169
- Running activities: 107
- Date range: 2025-06-14 - 2026-07-11

## Data-Quality Issues

- Missing values: none
- Columns stored as text:
- Duplicates: none
- Possible unit or formatting issues: date column having date and time

## Cleaning Tasks for the Next Stage

- 
- 